# Autonomous Code Writer

**Multi-Agent GitHub Workflow** | LangGraph + LangChain | AI Agents Course Project

---

## What it does

You give it a GitHub repo URL. It clones the repo, analyzes it, suggests a useful feature, **asks you** if the idea is good, creates a GitHub issue + branch, implements the change, runs tests, **asks you again** if the result looks right, opens a pull request, and reviews its own diff — leaving the final merge decision to you.

## Two human approval gates

| Gate | When | Your options |
|------|------|--------------|
| **1** | After feature proposal | `approve` → create issue & branch, `reject` → stop, `revise` → regenerate proposal |
| **2** | After implementation | `approve` → push branch & open PR, `reject` → stop, `revise` → redo implementation |

## Five agents

1. **Repository Analyst** — clones & inspects the repo (language, framework, tests)
2. **Feature Strategist** — proposes a conservative, useful feature
3. **GitHub Coordinator** — creates issues, branches, PRs, comments
4. **Implementation Agent** — makes focused code changes on the branch
5. **PR Review Agent** — self-reviews the diff and posts a GitHub comment

## Safety rules

- Never pushes to the default branch
- Never merges pull requests
- Never creates GitHub artifacts without your approval
- API keys loaded from env / Colab Secrets only

## Architecture

```
Request → Clone/Analyze → Propose → [HITL 1] → Issue+Branch → Implement → Verify → [HITL 2] → PR → Review → Done
```

## Assignment checklist

| Requirement | Status |
|-------------|--------|
| ≥2 agents | 5 agents |
| LangGraph stateful workflow | StateGraph + checkpointing |
| LangChain model/tools | OpenAI + tool wrappers |
| Multiple tools | Clone, inspect, GitHub API, test/lint |
| Conversational memory | Memory checkpointer |
| Human-in-the-Loop | 2 approval gates with revise loops |
| `execute_workflow(request)` | Public function |
| 5 test cases | 5 distinct calls on 1 demo repo |
| Approval test | Happy path |
| Revision test | Feature + implementation revision |

---

**Full details:** [README.md](https://github.com/BozhidarN7/autonomous-code-writer/blob/main/README.md) — architecture diagram, state schema, agent breakdown, requirements mapping, safety docs, Colab setup guide.

## ‼️Please read before proceeding‼️
In the submitted archive, you will find a .txt file containing a GitHub access token. The token is provided to ease the testing process. It has just enough access to run the notebook code. The token is restricted to the repository listed below.

**Feel free to use the token and the repository hardcoded below (Recommended).**

If you decide to use your own token, scroll to the bottom of the notebook. There you will find instructions and the permissions you need to configure.

**For LangSmith if you don't have a real api key just set a google colab env with random value. See the third code cell below.**

In [ ]:
!pip install -q langchain langgraph langchain-openai openai pygithub gitpython langsmith


In [ ]:
# Intended for local development only. Not needed in google colab
!pip install -q python-dotenv
from dotenv import load_dotenv

# Load .env file
load_dotenv()

In [ ]:
import os
import sys

# Attempt to load secrets from Google Colab Secrets (preferred in Colab environment)
try:
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY") or "")
    os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
    os.environ.setdefault("LANGSMITH_API_KEY", userdata.get("LANGSMITH_API_KEY") or "")
except ImportError:
    pass  # Not running in Colab; rely on environment variables

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()

# Set LangSmith tracking
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://aws.api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "autonomous-code-writer")

LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "").strip()
if LANGSMITH_API_KEY:
    os.environ.setdefault("LANGSMITH_API_KEY", LANGSMITH_API_KEY)

# Validate secrets without exposing values
missing = []
if not OPENAI_API_KEY:
    missing.append("OPENAI_API_KEY")
if not GITHUB_TOKEN:
    missing.append("GITHUB_TOKEN")

if missing:
    print("ERROR: Missing required secrets. Please set them via environment variables or Colab Secrets.")
    for m in missing:
        print(f"  - {m} is not set")
    print("Setup cannot continue until the secrets are configured.")
    sys.exit(1)
else:
    print("Secrets loaded successfully. (Values are hidden for security.)")


In [ ]:
from langchain_openai import ChatOpenAI

DEFAULT_MODEL_NAME = os.environ.get('OPENAI_MODEL_NAME', 'gpt-5.4-mini-2026-03-17')

llm = ChatOpenAI(
    model=DEFAULT_MODEL_NAME,
    temperature=0.2,
    openai_api_key=OPENAI_API_KEY,
)

print(f'OpenAI model initialized: {DEFAULT_MODEL_NAME}')

In [ ]:
from github import Github

github_client = Github(GITHUB_TOKEN, timeout=30)

try:
    user = github_client.get_user()
    login = user.login
    print(f'GitHub client authenticated as: {login}')
except Exception as e:
    print(f'ERROR: GitHub authentication failed — {e}')
    sys.exit(1)

# Verify repository access with a lightweight metadata call (optional sanity check)
# Replace with your demo repo if you want to test access immediately:
repo = github_client.get_repo('BozhidarN7/ds-algo-visualizer')
print(f'Repo access OK: {repo.full_name}')

In [ ]:
import re
import os
import tempfile
import shutil
from pathlib import Path
from git import Repo
from typing import Dict, List, Any, Optional

# Regex to extract owner/repo from a GitHub URL
GITHUB_URL_RE = re.compile(
    r'https?://github\.com/([A-Za-z0-9_.-]+)/([A-Za-z0-9_.-]+)'
)

# Directories to skip when walking the repo tree
SKIP_DIRS = {
    '.git',
    'node_modules',
    '__pycache__',
    '.venv', 'venv', 'env', 'virtualenv',
    'build', 'dist', 'target', 'out',
    '.pytest_cache', '.mypy_cache', '.tox',
    '.idea', '.vscode',
    'coverage', 'htmlcov',
}

# File extensions to language mapping
EXT_TO_LANGUAGE = {
    '.py': 'Python',
    '.js': 'JavaScript', '.ts': 'TypeScript',
    '.jsx': 'JavaScript (React)', '.tsx': 'TypeScript (React)',
    '.java': 'Java', '.kt': 'Kotlin',
    '.go': 'Go', '.rs': 'Rust',
    '.rb': 'Ruby', '.php': 'PHP',
    '.cpp': 'C++', '.c': 'C', '.h': 'C/C++',
    '.cs': 'C#', '.swift': 'Swift',
    '.r': 'R', '.scala': 'Scala',
    '.html': 'HTML', '.css': 'CSS', '.scss': 'SCSS',
    '.json': 'JSON', '.yaml': 'YAML', '.yml': 'YAML',
    '.md': 'Markdown',
    '.sh': 'Shell', '.sql': 'SQL',
}

# Package manager files to ecosystem mapping
PACKAGE_MANAGER_FILES = {
    'requirements.txt': 'pip', 'setup.py': 'setuptools', 'pyproject.toml': 'modern-python',
    'Pipfile': 'pipenv', 'poetry.lock': 'poetry',
    'package.json': 'npm/yarn/pnpm', 'package-lock.json': 'npm', 'yarn.lock': 'yarn', 'pnpm-lock.yaml': 'pnpm',
    'Cargo.toml': 'cargo', 'Cargo.lock': 'cargo',
    'go.mod': 'go modules', 'go.sum': 'go modules',
    'Gemfile': 'bundler', 'Gemfile.lock': 'bundler',
    'pom.xml': 'maven', 'build.gradle': 'gradle',
    'composer.json': 'composer',
    'pubspec.yaml': 'pub',
    'Podfile': 'cocoapods',
    'Makefile': 'make',
    'CMakeLists.txt': 'cmake',
    'Dockerfile': 'docker', 'docker-compose.yml': 'docker-compose',
}

# Test command detectors by ecosystem
TEST_COMMAND_HINTS = {
    'pip': ['pytest', 'python -m pytest'],
    'setuptools': ['pytest', 'python setup.py test'],
    'modern-python': ['pytest', 'python -m pytest'],
    'pipenv': ['pipenv run pytest'],
    'poetry': ['poetry run pytest'],
    'npm/yarn/pnpm': ['npm test', 'yarn test', 'pnpm test'],
    'npm': ['npm test'],
    'yarn': ['yarn test'],
    'pnpm': ['pnpm test'],
    'cargo': ['cargo test'],
    'go modules': ['go test ./...'],
    'bundler': ['bundle exec rspec', 'bundle exec minitest'],
    'maven': ['mvn test'],
    'gradle': ['gradle test'],
    'composer': ['vendor/bin/phpunit'],
    'pub': ['flutter test', 'dart test'],
    'make': ['make test'],
    'docker': ['docker build .'],
    'docker-compose': ['docker-compose build'],
}

# Lint command detectors by ecosystem
LINT_COMMAND_HINTS = {
    'pip': ['flake8', 'pylint', 'black --check .', 'ruff check .'],
    'setuptools': ['flake8', 'pylint'],
    'modern-python': ['ruff check .', 'black --check .', 'flake8'],
    'pipenv': ['pipenv run flake8'],
    'poetry': ['poetry run ruff check .'],
    'npm/yarn/pnpm': ['npm run lint', 'yarn lint', 'pnpm lint'],
    'npm': ['npm run lint'],
    'yarn': ['yarn lint'],
    'pnpm': ['pnpm lint'],
    'cargo': ['cargo clippy', 'cargo fmt --check'],
    'go modules': ['gofmt -l .', 'golangci-lint run'],
    'bundler': ['rubocop'],
    'maven': ['mvn verify'],
    'gradle': ['gradle check'],
    'composer': ['vendor/bin/phpcs'],
    'pub': ['flutter analyze', 'dart analyze'],
    'make': ['make lint'],
}


In [ ]:
def parse_repository_request(user_request: str) -> Optional[Dict[str, str]]:
    """Extract owner, repo name, and full URL from a user request."""
    match = GITHUB_URL_RE.search(user_request)
    if not match:
        return None
    owner, repo_name = match.group(1), match.group(2)
    repo_url = f'https://github.com/{owner}/{repo_name}'
    return {
        'owner': owner,
        'repo_name': repo_name,
        'repo_url': repo_url,
    }


def clone_repository(repo_url: str, owner: str, repo_name: str) -> str:
    """Clone a repository into a temporary directory. Returns the local path."""
    base_tmp = tempfile.gettempdir()
    clone_dir = os.path.join(base_tmp, f'acw_{owner}_{repo_name}')
    if os.path.exists(clone_dir):
        shutil.rmtree(clone_dir)
    Repo.clone_from(repo_url, clone_dir, depth=1)
    return clone_dir


def list_repository_files(clone_dir: str, max_files: int = 200) -> List[str]:
    """Walk the cloned repo and return relative file paths, skipping irrelevant dirs."""
    files = []
    root = Path(clone_dir)
    for path in root.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(root)
        parts = set(rel.parts[:-1])
        if parts & SKIP_DIRS:
            continue
        if any(part in SKIP_DIRS for part in rel.parts):
            continue
        files.append(str(rel))
        if len(files) >= max_files:
            break
    return sorted(files)


def detect_project_metadata(files: List[str]) -> Dict[str, Any]:
    """Detect language, package manager, and likely test/lint commands from file list."""
    metadata = {
        'languages': set(),
        'package_managers': set(),
        'test_commands': set(),
        'lint_commands': set(),
        'key_files': [],
    }

    for f in files:
        name = os.path.basename(f)
        if name in PACKAGE_MANAGER_FILES:
            metadata['package_managers'].add(PACKAGE_MANAGER_FILES[name])
            metadata['key_files'].append(f)
        _, ext = os.path.splitext(f)
        if ext in EXT_TO_LANGUAGE:
            metadata['languages'].add(EXT_TO_LANGUAGE[ext])
        if name in ('README.md', 'CONTRIBUTING.md', 'LICENSE', '.gitignore', 'Makefile', 'Dockerfile', 'docker-compose.yml', 'tox.ini', 'setup.cfg', 'pytest.ini'):
            if f not in metadata['key_files']:
                metadata['key_files'].append(f)

    for pm in metadata['package_managers']:
        if pm in TEST_COMMAND_HINTS:
            metadata['test_commands'].update(TEST_COMMAND_HINTS[pm])
        if pm in LINT_COMMAND_HINTS:
            metadata['lint_commands'].update(LINT_COMMAND_HINTS[pm])

    metadata['languages'] = sorted(metadata['languages'])
    metadata['package_managers'] = sorted(metadata['package_managers'])
    metadata['test_commands'] = sorted(metadata['test_commands'])
    metadata['lint_commands'] = sorted(metadata['lint_commands'])
    metadata['key_files'] = metadata['key_files'][:20]
    return metadata


def read_key_file_snippets(clone_dir: str, key_files: List[str], max_lines: int = 50) -> Dict[str, str]:
    """Read the first max_lines of each key file for agent context."""
    snippets = {}
    for f in key_files:
        path = Path(clone_dir) / f
        if not path.exists():
            continue
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
                lines = []
                for i, line in enumerate(fh):
                    if i >= max_lines:
                        break
                    lines.append(line.rstrip('\n'))
                snippets[f] = '\n'.join(lines)
        except Exception:
            continue
    return snippets


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

REPOSITORY_ANALYST_PROMPT = """
You are a Repository Analyst. Your job is to understand a codebase from its file tree, metadata, and key file snippets, then produce a concise technical summary.

Guidelines:
- Identify the primary language(s) and framework(s).
- Note the project structure (e.g., monolith, library, service, CLI tool, web app).
- Mention the package manager and build/test setup if detectable.
- List the most important directories and their purposes.
- Flag anything unusual or noteworthy (e.g., no tests, unconventional layout, multiple languages).
- Keep the summary under 250 words.
- Do not include secrets, credentials, or sensitive data.
"""

def build_repo_analysis_prompt(repo_owner: str, repo_name: str, files: List[str], metadata: Dict[str, Any], snippets: Dict[str, str]) -> str:
    lines = [
        f'Repository: {repo_owner}/{repo_name}',
        f'Total files scanned (sample): {len(files)}',
        '',
        'Languages detected:',
    ]
    for lang in metadata.get('languages', []):
        lines.append(f'  - {lang}')
    lines.append('')
    lines.append('Package managers / build tools:')
    for pm in metadata.get('package_managers', []):
        lines.append(f'  - {pm}')
    lines.append('')
    lines.append('Likely test commands:')
    for cmd in metadata.get('test_commands', []):
        lines.append(f'  - {cmd}')
    lines.append('')
    lines.append('Likely lint commands:')
    for cmd in metadata.get('lint_commands', []):
        lines.append(f'  - {cmd}')
    lines.append('')
    lines.append('Key files:')
    for kf in metadata.get('key_files', []):
        lines.append(f'  - {kf}')
    lines.append('')
    lines.append('File tree sample (first 50 files):')
    for f in files[:50]:
        lines.append(f'  {f}')
    if snippets:
        lines.append('')
        lines.append('Key file snippets:')
        for fname, content in snippets.items():
            lines.append(f'\n--- {fname} ---')
            lines.append(content[:800])
    return '\n'.join(lines)


def run_repository_analyst(repo_owner: str, repo_name: str, files: List[str], metadata: Dict[str, Any], snippets: Dict[str, str]) -> str:
    """Invoke the LLM to synthesize a repository summary."""
    prompt = build_repo_analysis_prompt(repo_owner, repo_name, files, metadata, snippets)
    messages = [
        SystemMessage(content=REPOSITORY_ANALYST_PROMPT),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    return str(response.content)


In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from typing import Annotated
from langgraph.graph.message import add_messages

class WorkflowState(TypedDict, total=False):
    user_request: str
    repo_url: str
    repo_owner: str
    repo_name: str
    repo_clone_dir: str
    repo_files: List[str]
    repo_metadata: Dict[str, Any]
    repo_summary: str
    feature_proposal: Dict[str, Any]
    human_decision_gate1: Dict[str, Any]
    issue_url: str
    issue_number: int
    branch_name: str
    pr_title: str
    pr_body: str
    implementation_plan: str
    changed_files: List[str]
    diff_summary: str
    verification_results: Dict[str, Any]
    human_decision_gate2: Dict[str, Any]
    pr_url: str
    pr_number: int
    review_summary: str
    review_comment_url: str
    messages: Annotated[List[BaseMessage], add_messages]
    error: str


In [ ]:
def parse_request_node(state: WorkflowState) -> WorkflowState:
    """Parse the user request and extract repository information."""
    user_request = state.get('user_request', '')
    parsed = parse_repository_request(user_request)
    if not parsed:
        return {
            **state,
            'error': 'No valid GitHub repository URL found in the request.',
        }
    return {
        **state,
        'repo_url': parsed['repo_url'],
        'repo_owner': parsed['owner'],
        'repo_name': parsed['repo_name'],
        'messages': state.get('messages', []) + [
            HumanMessage(content=f'Parsed repository: {parsed["owner"]}/{parsed["repo_name"]}')
        ],
    }


def clone_and_inspect_node(state: WorkflowState) -> WorkflowState:
    """Clone the repository and inspect its file tree and metadata."""
    repo_url = state.get('repo_url')
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    if not repo_url or not owner or not repo_name:
        return {**state, 'error': 'Missing repository information for cloning.'}
    try:
        clone_dir = clone_repository(repo_url, owner, repo_name)
        files = list_repository_files(clone_dir)
        metadata = detect_project_metadata(files)
        snippets = read_key_file_snippets(clone_dir, metadata.get('key_files', []))
        return {
            **state,
            'repo_clone_dir': clone_dir,
            'repo_files': files,
            'repo_metadata': metadata,
            'messages': state.get('messages', []) + [
                HumanMessage(content=f'Cloned {len(files)} files into {clone_dir}')
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Failed to clone or inspect repository: {e}',
        }


def analyze_repository_node(state: WorkflowState) -> WorkflowState:
    """Run the Repository Analyst Agent to produce a summary."""
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    files = state.get('repo_files', [])
    metadata = state.get('repo_metadata', {})
    clone_dir = state.get('repo_clone_dir', '')
    if not files or not metadata:
        return {**state, 'error': 'No repository files or metadata available for analysis.'}
    snippets = read_key_file_snippets(clone_dir, metadata.get('key_files', []))
    try:
        summary = run_repository_analyst(owner, repo_name, files, metadata, snippets)
        return {
            **state,
            'repo_summary': summary,
            'messages': state.get('messages', []) + [
                HumanMessage(content='Repository Analyst produced a summary.')
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Repository analysis failed: {e}',
        }


In [ ]:
# Demo: run the repository analysis slice end-to-end on a real repository
# Replace the URL with your own demo repository when running for real.

DEMO_REPO_URL = 'https://github.com/BozhidarN7/ds-algo-visualizer'

demo_state: WorkflowState = {
    'user_request': f'Analyze this repository and suggest a useful improvement: {DEMO_REPO_URL}',
    'messages': [],
}

# Step 1: parse request
demo_state = parse_request_node(demo_state)
if demo_state.get('error'):
    print('Parse error:', demo_state['error'])
else:
    print(f"Parsed: {demo_state['repo_owner']}/{demo_state['repo_name']}")
    print(f"URL: {demo_state['repo_url']}")

# Step 2: clone and inspect
demo_state = clone_and_inspect_node(demo_state)
if demo_state.get('error'):
    print('Clone/inspect error:', demo_state['error'])
else:
    print(f"Cloned to: {demo_state['repo_clone_dir']}")
    print(f"Files found: {len(demo_state['repo_files'])}")
    meta = demo_state['repo_metadata']
    print(f"Languages: {meta.get('languages')}")
    print(f"Package managers: {meta.get('package_managers')}")
    print(f"Test commands: {meta.get('test_commands')}")
    print(f"Lint commands: {meta.get('lint_commands')}")
    print(f"Key files: {meta.get('key_files')}")

# Step 3: analyze with LLM
demo_state = analyze_repository_node(demo_state)
if demo_state.get('error'):
    print('Analysis error:', demo_state['error'])
else:
    print('\n--- Repository Summary ---\n')
    print(demo_state['repo_summary'])

    print('\n--- Detected Stack ---\n')
    meta = demo_state['repo_metadata']
    print(f"Languages: {', '.join(meta.get('languages', [])) or 'None detected'}")
    print(f"Package managers: {', '.join(meta.get('package_managers', [])) or 'None detected'}")
    print(f"Likely tests: {', '.join(meta.get('test_commands', [])) or 'None detected'}")
    print(f"Likely lint: {', '.join(meta.get('lint_commands', [])) or 'None detected'}")


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

FEATURE_STRATEGIST_PROMPT = """
You are a Feature Strategist. Your job is to propose a conservative, useful feature for a repository based on its technical summary and metadata.

Guidelines:
- Generate 2-3 feature ideas relevant to the repository's stack and purpose.
- Prefer small-to-medium features: documentation improvements, developer-experience enhancements, new tests, examples, or focused code additions.
- Avoid risky changes: no authentication rewrites, no database migrations, no deployment changes, no broad dependency upgrades.
- Select the best option and produce a structured proposal.

Output format (strict JSON):
{
  "ideas": ["idea 1", "idea 2", "idea 3"],
  "selected_index": 0,
  "title": "Concise issue title",
  "body": "Detailed feature description, goal, and motivation.",
  "value": "Why this feature is useful.",
  "implementation_scope": "What files or areas will likely change.",
  "risk_level": "low|medium|high",
  "acceptance_criteria": ["criterion 1", "criterion 2"]
}
"""

def build_feature_strategy_prompt(repo_owner: str, repo_name: str, repo_summary: str, metadata: Dict[str, Any]) -> str:
    lines = [
        f'Repository: {repo_owner}/{repo_name}',
        '',
        'Repository summary:',
        repo_summary,
        '',
        'Detected languages:',
    ]
    for lang in metadata.get('languages', []):
        lines.append(f'  - {lang}')
    lines.append('')
    lines.append('Package managers / build tools:')
    for pm in metadata.get('package_managers', []):
        lines.append(f'  - {pm}')
    lines.append('')
    lines.append('Likely test commands:')
    for cmd in metadata.get('test_commands', []):
        lines.append(f'  - {cmd}')
    lines.append('')
    lines.append('Likely lint commands:')
    for cmd in metadata.get('lint_commands', []):
        lines.append(f'  - {cmd}')
    return '\n'.join(lines)


import json

def run_feature_strategist(repo_owner: str, repo_name: str, repo_summary: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    prompt = build_feature_strategy_prompt(repo_owner, repo_name, repo_summary, metadata)
    messages = [
        SystemMessage(content=FEATURE_STRATEGIST_PROMPT),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    content = str(response.content)

    cleaned = content.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned[7:]
    if cleaned.startswith('```'):
        cleaned = cleaned[3:]
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        proposal = json.loads(cleaned)
    except json.JSONDecodeError:
        proposal = {
            'ideas': ['Feature suggestion based on repository analysis'],
            'selected_index': 0,
            'title': 'Improve repository based on analysis',
            'body': content,
            'value': 'Improves the repository',
            'implementation_scope': 'TBD',
            'risk_level': 'medium',
            'acceptance_criteria': ['Change is implemented and tested'],
        }
    return proposal


In [ ]:
def generate_feature_proposal_node(state: WorkflowState) -> WorkflowState:
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    summary = state.get('repo_summary', '')
    metadata = state.get('repo_metadata', {})
    if not summary:
        return {**state, 'error': 'No repository summary available for feature strategy.'}
    try:
        proposal = run_feature_strategist(owner, repo_name, summary, metadata)
        return {
            **state,
            'feature_proposal': proposal,
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"Feature Strategist selected: {proposal.get('title', 'Untitled')}")
            ],
        }
    except Exception as e:
        return {**state, 'error': f'Feature strategy failed: {e}'}


def human_review_gate1_node(state: WorkflowState) -> WorkflowState:
    proposal = state.get('feature_proposal', {})
    title = proposal.get('title', 'Untitled')
    body = proposal.get('body', '')
    value = proposal.get('value', '')
    scope = proposal.get('implementation_scope', '')
    risk = proposal.get('risk_level', 'unknown')
    criteria = proposal.get('acceptance_criteria', [])

    print('\n' + '=' * 60)
    print('FEATURE PROPOSAL — HUMAN REVIEW REQUIRED')
    print('=' * 60)
    print(f"Title: {title}")
    print(f"Risk: {risk}")
    print('\nDescription:')
    print(body[:600] + ('...' if len(body) > 600 else ''))
    print('\nValue:')
    print(value[:300] + ('...' if len(value) > 300 else ''))
    print('\nImplementation scope:')
    print(scope[:300] + ('...' if len(scope) > 300 else ''))
    print('\nAcceptance criteria:')
    for c in criteria:
        print(f'  - {c}')
    print('\n' + '-' * 60)
    print('Type your decision:')
    print('  approve  — create GitHub issue and branch')
    print('  reject   — stop the workflow')
    print('  revise <feedback> — regenerate proposal with your feedback')
    print('=' * 60 + '\n')

    try:
        user_input = input('Your decision: ').strip()
    except EOFError:
        user_input = ''

    decision = 'pending'
    feedback = ''
    if user_input.lower() == 'approve':
        decision = 'approved'
    elif user_input.lower() == 'reject':
        decision = 'rejected'
    elif user_input.lower().startswith('revise'):
        decision = 'revise'
        feedback = user_input[6:].strip()
    else:
        decision = 'revise'
        feedback = user_input

    return {
        **state,
        'human_decision_gate1': {
            'decision': decision,
            'feedback': feedback,
            'timestamp': None,
        },
        'messages': state.get('messages', []) + [
            HumanMessage(content=f'Human gate 1 decision: {decision}')
        ],
    }


def revise_feature_proposal_node(state: WorkflowState) -> WorkflowState:
    proposal = state.get('feature_proposal', {})
    feedback = state.get('human_decision_gate1', {}).get('feedback', '')
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    summary = state.get('repo_summary', '')
    metadata = state.get('repo_metadata', {})

    revision_prompt = f"""
The previous feature proposal was:

Title: {proposal.get('title', '')}
Body: {proposal.get('body', '')}

Human feedback for revision:
{feedback}

Please regenerate the feature proposal incorporating this feedback.
"""

    messages = [
        SystemMessage(content=FEATURE_STRATEGIST_PROMPT),
        HumanMessage(content=revision_prompt),
    ]
    response = llm.invoke(messages)
    content = str(response.content)

    cleaned = content.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned[7:]
    if cleaned.startswith('```'):
        cleaned = cleaned[3:]
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        proposal = json.loads(cleaned)
    except json.JSONDecodeError:
        proposal = {
            'ideas': ['Revised feature suggestion'],
            'selected_index': 0,
            'title': 'Revised feature proposal',
            'body': content,
            'value': 'Improves the repository',
            'implementation_scope': 'TBD',
            'risk_level': 'medium',
            'acceptance_criteria': ['Change is implemented and tested'],
        }
    return {
        **state,
        'feature_proposal': proposal,
        'human_decision_gate1': {'decision': 'pending', 'feedback': '', 'timestamp': None},
        'messages': state.get('messages', []) + [
            HumanMessage(content=f"Feature Strategist revised proposal: {proposal.get('title', 'Untitled')}")
        ],
    }


def route_gate1_decision(state: WorkflowState) -> str:
    decision = state.get('human_decision_gate1', {}).get('decision', 'pending')
    if decision == 'approved':
        return 'approve'
    elif decision == 'rejected':
        return 'reject'
    else:
        return 'revise'


In [ ]:
# Demo: Feature Strategist + HITL Gate 1
# This cell demonstrates the feature proposal generation and human approval flow.
# In a real run, the graph would pause at gate 1 for human input.

demo_state2: WorkflowState = {
    'user_request': f'Analyze this repository and suggest a useful improvement: {DEMO_REPO_URL}',
    'messages': [],
}

demo_state2 = parse_request_node(demo_state2)
demo_state2 = clone_and_inspect_node(demo_state2)
demo_state2 = analyze_repository_node(demo_state2)

if demo_state2.get('error'):
    print('Error before feature strategy:', demo_state2['error'])
else:
    demo_state2 = generate_feature_proposal_node(demo_state2)
    proposal = demo_state2.get('feature_proposal', {})
    print('\n--- Generated Feature Proposal ---\n')
    print(f"Title: {proposal.get('title')}")
    print(f"Risk: {proposal.get('risk_level')}")
    print(f"Value: {proposal.get('value')}")
    print(f"Scope: {proposal.get('implementation_scope')}")
    print('\nAcceptance criteria:')
    for c in proposal.get('acceptance_criteria', []):
        print(f'  - {c}')
    print('\n--- Full Body ---\n')
    print(proposal.get('body', '')[:800] + ('...' if len(proposal.get('body', '')) > 800 else ''))

    print('\n--- HITL Gate 1 Simulation ---\n')
    print('In the full workflow, the graph would pause here and ask for:')
    print('  approve  — proceed to create issue and branch')
    print('  reject   — stop the workflow')
    print('  revise <feedback> — regenerate the proposal')


In [ ]:
import re
import os
from git import Repo
from github import GithubException
from typing import Dict, Any

def create_github_issue(owner: str, repo_name: str, title: str, body: str) -> Dict[str, Any]:
    """Create a GitHub issue and return its URL and number."""
    repo = github_client.get_repo(f'{owner}/{repo_name}')
    issue = repo.create_issue(title=title, body=body)
    return {
        'issue_url': issue.html_url,
        'issue_number': issue.number,
    }

def generate_safe_branch_name(issue_number: int, feature_title: str) -> str:
    """Generate a branch name that references the issue and feature."""
    slug = re.sub(r'[^a-zA-Z0-9]+', '-', feature_title.lower()).strip('-')
    slug = re.sub(r'-+', '-', slug)[:40]
    return f'feature/acw-{issue_number}-{slug}'

def create_implementation_branch(clone_dir: str, branch_name: str) -> str:
    """Create a new local branch based on the default branch. Returns the default branch name."""
    repo = Repo(clone_dir)
    default_branch = repo.active_branch.name
    new_branch = repo.create_head(branch_name)
    new_branch.checkout()
    return default_branch

def push_branch(clone_dir: str, branch_name: str, repo_url: str, token: str) -> None:
    """Push the local branch to the remote repository."""
    if not token:
        raise ValueError('GITHUB_TOKEN is required to push a branch.')

    repo = Repo(clone_dir)
    auth_url = repo_url.replace('https://', f'https://x-access-token:{token}@')
    push_output = repo.git.push(auth_url, f'{branch_name}:{branch_name}')
    if '[rejected]' in push_output.lower() or 'error' in push_output.lower():
        raise RuntimeError(push_output)


def create_pull_request(owner: str, repo_name: str, title: str, body: str, head_branch: str, base_branch: str = 'main') -> Dict[str, Any]:
    """Create a GitHub pull request and return its URL and number."""
    repo = github_client.get_repo(f'{owner}/{repo_name}')
    try:
        pr = repo.create_pull(
            title=title,
            body=body,
            head=head_branch,
            base=base_branch,
        )
        return {
            'pr_url': pr.html_url,
            'pr_number': pr.number,
        }
    except GithubException as e:
        error_text = str(e.data) if getattr(e, 'data', None) else str(e)
        if e.status == 422 and 'already exists' in error_text.lower():
            pulls = repo.get_pulls(head=f'{owner}:{head_branch}', state='open')
            for existing in pulls:
                return {
                    'pr_url': existing.html_url,
                    'pr_number': existing.number,
                }
        raise


In [ ]:
def create_issue_node(state: WorkflowState) -> WorkflowState:
    """Create a GitHub issue from the approved feature proposal."""
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    proposal = state.get('feature_proposal', {})
    title = proposal.get('title', 'Feature proposal')
    body_parts = [
        proposal.get('body', ''),
        '',
        '## Value',
        proposal.get('value', ''),
        '',
        '## Implementation Scope',
        proposal.get('implementation_scope', ''),
        '',
        '## Risk Level',
        proposal.get('risk_level', 'unknown'),
        '',
        '## Acceptance Criteria',
    ]
    for criterion in proposal.get('acceptance_criteria', []):
        body_parts.append(f'- {criterion}')
    body_parts.append('')
    body_parts.append('---')
    body_parts.append('*This issue was created by the Autonomous Code Writer workflow.*')
    body_parts.append('*The human user retains final authority over merge decisions.*')
    body = '\n'.join(body_parts)
    try:
        result = create_github_issue(owner, repo_name, title, body)
        return {
            **state,
            'issue_url': result['issue_url'],
            'issue_number': result['issue_number'],
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"Created GitHub issue #{result['issue_number']}: {result['issue_url']}")
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Failed to create GitHub issue: {e}',
        }

def create_branch_node(state: WorkflowState) -> WorkflowState:
    """Create a local implementation branch based on the default branch."""
    clone_dir = state.get('repo_clone_dir')
    issue_number = state.get('issue_number')
    proposal = state.get('feature_proposal', {})
    if not clone_dir or not issue_number:
        return {
            **state,
            'error': 'Missing clone directory or issue number for branch creation.',
        }
    try:
        branch_name = generate_safe_branch_name(issue_number, proposal.get('title', 'feature'))
        default_branch = create_implementation_branch(clone_dir, branch_name)
        return {
            **state,
            'branch_name': branch_name,
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"Created implementation branch: {branch_name} (based on {default_branch})")
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Failed to create implementation branch: {e}',
        }


In [ ]:
import json
import os
from pathlib import Path
from git import Repo
from langchain_core.messages import SystemMessage, HumanMessage

IMPLEMENTATION_AGENT_PROMPT = """
You are an Implementation Agent. Your job is to implement a feature on a local branch by producing a structured plan and then generating file-level changes.

Safety rules (absolute):
- Never modify files outside the repository clone directory.
- Never modify .git/, .env, secrets files, CI/CD configs, or deployment manifests.
- Never rewrite authentication, database migrations, or broad dependency upgrades.
- Never delete files unless the approved issue explicitly requires it.
- Prefer documentation, tests, examples, or small focused code changes over risky architecture changes.
- Keep changes focused on the approved issue; avoid unrelated refactors.
- Add or update tests when feasible for the selected change.

Output format (strict JSON):
{
  "plan": "Concise implementation plan describing what will change and why.",
  "changed_files": [
    {
      "path": "relative/path/from/repo/root",
      "action": "create|modify|append",
      "description": "What this file change does.",
      "content": "Full new file content for create/modify; text to append for append."
    }
  ]
}

Guidelines:
- For 'create', provide the complete file content.
- For 'modify', provide the complete new file content (the agent will overwrite the file). You MUST inspect the existing file content provided in the prompt and produce a coherent updated version. Do NOT output placeholder text like 'REPLACE_WITH_FULL_NEW_FILE_CONTENT_AFTER_INSPECTION'.
- For 'append', provide only the text to append at the end of the existing file.
- Use only relative paths from the repository root.
- If tests are feasible, include a test file change.
- If you are modifying an existing file, read the existing content carefully and produce the full updated content. Never emit partial content or placeholders.
"""

SAFETY_BLOCKED_PATHS = {
    '.git', '.env', '.env.local', '.env.production', '.env.staging',
    'secrets', 'credentials', 'token', 'key.pem', 'id_rsa',
}

SAFETY_BLOCKED_EXTENSIONS = {
    '.key', '.pem', '.p12', '.pfx', '.crt', '.cer',
}

SAFETY_BLOCKED_DIR_NAMES = {
    '.git', '.github/workflows', '.circleci', '.travis', '.jenkins',
    'deploy', 'deployment', 'k8s', 'kubernetes', 'helm',
    'terraform', 'ansible', 'infrastructure',
}


def is_path_safe(repo_clone_dir: str, rel_path: str) -> bool:
    """Check whether a relative path is safe to modify."""
    parts = Path(rel_path).parts
    name = os.path.basename(rel_path)
    _, ext = os.path.splitext(name.lower())

    for part in parts:
        if part in SAFETY_BLOCKED_PATHS:
            return False
    for blocked_dir in SAFETY_BLOCKED_DIR_NAMES:
        if blocked_dir in parts:
            return False
    if name.lower() in SAFETY_BLOCKED_PATHS:
        return False
    if ext in SAFETY_BLOCKED_EXTENSIONS:
        return False
    resolved = Path(repo_clone_dir).resolve() / rel_path
    try:
        resolved.relative_to(Path(repo_clone_dir).resolve())
    except ValueError:
        return False
    return True


def _read_file_content(repo_clone_dir: str, rel_path: str, max_lines: int = 150) -> str:
    """Read existing file content for modify actions."""
    target = Path(repo_clone_dir) / rel_path
    if not target.exists():
        return ''
    try:
        with open(target, 'r', encoding='utf-8', errors='ignore') as fh:
            lines = []
            for i, line in enumerate(fh):
                if i >= max_lines:
                    break
                lines.append(line.rstrip('\n'))
            return '\n'.join(lines)
    except Exception:
        return ''


def build_implementation_prompt(repo_owner: str, repo_name: str, repo_summary: str, metadata: dict, feature_proposal: dict, repo_clone_dir: str, changed_files_spec: list = None, revision_feedback: str = '', previous_plan: str = '', previous_diff_summary: str = '') -> str:
    lines = [
        f'Repository: {repo_owner}/{repo_name}',
        '',
        'Repository summary:',
        repo_summary,
        '',
        'Detected languages:',
    ]
    for lang in metadata.get('languages', []):
        lines.append(f'  - {lang}')
    lines.append('')
    lines.append('Package managers:')
    for pm in metadata.get('package_managers', []):
        lines.append(f'  - {pm}')
    lines.append('')
    lines.append('Approved feature proposal:')
    lines.append(f"Title: {feature_proposal.get('title', '')}")
    lines.append(f"Body: {feature_proposal.get('body', '')}")
    lines.append(f"Scope: {feature_proposal.get('implementation_scope', '')}")
    lines.append(f"Risk: {feature_proposal.get('risk_level', 'unknown')}")
    lines.append('')
    lines.append('Existing file tree (first 80 files):')
    files = metadata.get('all_files', [])
    if not files:
        files = []
        root = Path(repo_clone_dir)
        for p in root.rglob('*'):
            if p.is_file() and not any(part in SKIP_DIRS for part in p.relative_to(root).parts):
                files.append(str(p.relative_to(root)))
                if len(files) >= 80:
                    break
    for f in files[:80]:
        lines.append(f'  {f}')
    if revision_feedback:
        lines.append('')
        lines.append('Human feedback for implementation revision:')
        lines.append(revision_feedback)
        if previous_plan:
            lines.append('')
            lines.append('Previous implementation plan:')
            lines.append(previous_plan)
        if previous_diff_summary:
            lines.append('')
            lines.append('Previous attempt result:')
            lines.append(previous_diff_summary[:2000])
        lines.append('')
        lines.append('Your revised response must address the human feedback with concrete code changes. Do not return an empty changed_files list for a feasible request.')
    if changed_files_spec:
        lines.append('')
        lines.append('Files you intend to modify (with existing content):')
        for entry in changed_files_spec:
            if entry.get('action') == 'modify':
                rel = entry.get('path', '')
                content = _read_file_content(repo_clone_dir, rel)
                lines.append(f"\n--- {rel} ---")
                lines.append(content[:2000] if content else '(file not found or empty)')
    return '\n'.join(lines)


def run_implementation_agent(repo_owner: str, repo_name: str, repo_summary: str, metadata: dict, feature_proposal: dict, repo_clone_dir: str, revision_feedback: str = '', previous_plan: str = '', previous_diff_summary: str = '') -> dict:
    """Generate an implementation plan and file changes."""
    prompt = build_implementation_prompt(
        repo_owner,
        repo_name,
        repo_summary,
        metadata,
        feature_proposal,
        repo_clone_dir,
        revision_feedback=revision_feedback,
        previous_plan=previous_plan,
        previous_diff_summary=previous_diff_summary,
    )
    messages = [
        SystemMessage(content=IMPLEMENTATION_AGENT_PROMPT),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    content = str(response.content)

    cleaned = content.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned[7:]
    if cleaned.startswith('```'):
        cleaned = cleaned[3:]
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        result = json.loads(cleaned)
    except json.JSONDecodeError:
        result = {
            'plan': 'Fallback plan: add a README note about the approved feature.',
            'changed_files': [
                {
                    'path': 'ACW_FEATURE_NOTE.md',
                    'action': 'create',
                    'description': 'Documentation note about the approved feature.',
                    'content': f"# Feature Note\n\nApproved feature: {feature_proposal.get('title', 'N/A')}\n\n{feature_proposal.get('body', '')}",
                }
            ],
        }
    changed_files = result.get('changed_files', [])
    if not isinstance(changed_files, list):
        changed_files = []
        result['changed_files'] = changed_files
    if not changed_files:
        retry_messages = messages + [
            HumanMessage(content='Your previous response returned no file changes. Re-evaluate the repository and return concrete file edits that implement the approved feature. If the request is infeasible, explain the blocker clearly in the plan.')
        ]
        retry_response = llm.invoke(retry_messages)
        retry_content = str(retry_response.content)
        retry_cleaned = retry_content.strip()
        if retry_cleaned.startswith('```json'):
            retry_cleaned = retry_cleaned[7:]
        if retry_cleaned.startswith('```'):
            retry_cleaned = retry_cleaned[3:]
        if retry_cleaned.endswith('```'):
            retry_cleaned = retry_cleaned[:-3]
        retry_cleaned = retry_cleaned.strip()
        try:
            retry_result = json.loads(retry_cleaned)
            result = retry_result
            changed_files = retry_result.get('changed_files', [])
            if not isinstance(changed_files, list):
                changed_files = []
            result['changed_files'] = changed_files
        except json.JSONDecodeError:
            changed_files = []
            result['changed_files'] = changed_files
    for entry in changed_files:
        if entry.get('action') == 'modify':
            rel = entry.get('path', '')
            existing = _read_file_content(repo_clone_dir, rel)
            if existing and 'REPLACE_WITH' in entry.get('content', ''):
                entry['content'] = existing
    return result


def apply_file_changes(repo_clone_dir: str, changed_files: list) -> list:
    """Apply structured file changes to the local clone. Returns list of changed relative paths."""
    applied = []
    for entry in changed_files:
        rel_path = entry.get('path', '')
        action = entry.get('action', '')
        content = entry.get('content', '')
        if not rel_path or not action:
            continue
        if not is_path_safe(repo_clone_dir, rel_path):
            print(f'SKIPPED unsafe path: {rel_path}')
            continue
        target = Path(repo_clone_dir) / rel_path
        if action in ('create', 'modify'):
            target.parent.mkdir(parents=True, exist_ok=True)
            with open(target, 'w', encoding='utf-8') as fh:
                fh.write(content)
            applied.append(rel_path)
        elif action == 'append':
            if target.exists():
                with open(target, 'a', encoding='utf-8') as fh:
                    fh.write('\n' + content)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with open(target, 'w', encoding='utf-8') as fh:
                    fh.write(content)
            applied.append(rel_path)
    return applied


def commit_changes(clone_dir: str, branch_name: str, message: str) -> None:
    """Stage and commit all changes on the current branch."""
    repo = Repo(clone_dir)
    if repo.is_dirty(untracked_files=True):
        repo.git.add(all=True)
        repo.index.commit(message)


def get_git_diff_summary(clone_dir: str) -> str:
    """Return a summary of changes (stat + diff) on the current branch vs default."""
    repo = Repo(clone_dir)
    default_branch = repo.git.symbolic_ref('refs/remotes/origin/HEAD').replace('refs/remotes/origin/', '')
    stat = repo.git.diff(f'origin/{default_branch}...{repo.active_branch.name}', '--stat')
    diff = repo.git.diff(f'origin/{default_branch}...{repo.active_branch.name}')
    max_diff_len = 4000
    if len(diff) > max_diff_len:
        diff = diff[:max_diff_len] + '\n... (truncated)\n'
    return f'Stat:\n{stat}\n\nDiff:\n{diff}'


def get_changed_files_list(clone_dir: str) -> list:
    """Return a list of changed file paths relative to repo root."""
    repo = Repo(clone_dir)
    default_branch = repo.git.symbolic_ref('refs/remotes/origin/HEAD').replace('refs/remotes/origin/', '')
    diff = repo.git.diff(f'origin/{default_branch}...{repo.active_branch.name}', name_only=True)
    return [f for f in diff.strip().split('\n') if f]


In [ ]:
def implementation_node(state: WorkflowState) -> WorkflowState:
    """Run the Implementation Agent to plan and apply changes."""
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    clone_dir = state.get('repo_clone_dir')
    summary = state.get('repo_summary', '')
    metadata = state.get('repo_metadata', {})
    proposal = state.get('feature_proposal', {})
    branch_name = state.get('branch_name', '')
    revision_feedback = state.get('human_decision_gate2', {}).get('feedback', '').strip()
    previous_plan = state.get('implementation_plan', '')
    previous_diff_summary = state.get('diff_summary', '')

    if not clone_dir or not branch_name:
        return {
            **state,
            'error': 'Missing clone directory or branch name for implementation.',
        }

    try:
        result = run_implementation_agent(
            owner,
            repo_name,
            summary,
            metadata,
            proposal,
            clone_dir,
            revision_feedback=revision_feedback,
            previous_plan=previous_plan,
            previous_diff_summary=previous_diff_summary,
        )
        plan = result.get('plan', '')
        changed_files_spec = result.get('changed_files', [])
        if not changed_files_spec:
            return {
                **state,
                'implementation_plan': plan,
                'changed_files': [],
                'diff_summary': 'Implementation agent returned no file changes. The workflow cannot continue to PR creation without a concrete diff.',
                'error': 'Implementation agent returned no file changes.',
                'human_decision_gate2': {'decision': 'pending', 'feedback': '', 'timestamp': None},
                'messages': state.get('messages', []) + [
                    HumanMessage(content='Implementation blocked: the agent returned no file changes.')
                ],
            }
        applied = apply_file_changes(clone_dir, changed_files_spec)

        if applied:
            commit_msg = f"feat: {proposal.get('title', 'Implementation')} (ACW)"
            commit_changes(clone_dir, branch_name, commit_msg)
            diff_summary = get_git_diff_summary(clone_dir)
            changed_files = get_changed_files_list(clone_dir)
            if not changed_files:
                return {
                    **state,
                    'implementation_plan': plan,
                    'changed_files': [],
                    'diff_summary': 'Implementation wrote files locally but produced no git diff against the base branch. The workflow cannot continue to PR creation without a concrete diff.',
                    'error': 'Implementation produced no git diff against the base branch.',
                    'human_decision_gate2': {'decision': 'pending', 'feedback': '', 'timestamp': None},
                    'messages': state.get('messages', []) + [
                        HumanMessage(content='Implementation blocked: no git diff was produced against the base branch.')
                    ],
                }
        else:
            diff_summary = 'No file changes were applied.'
            changed_files = []

        return {
            **state,
            'implementation_plan': plan,
            'changed_files': changed_files,
            'diff_summary': diff_summary,
            'human_decision_gate2': {'decision': 'pending', 'feedback': '', 'timestamp': None},
            'messages': state.get('messages', []) + [
                HumanMessage(content=f'Implementation Agent applied {len(applied)} file change(s).')
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Implementation failed: {e}',
        }


In [ ]:
def verification_node(state: WorkflowState) -> WorkflowState:
    """Run test and lint verification on the implementation branch."""
    clone_dir = state.get('repo_clone_dir')
    metadata = state.get('repo_metadata', {})
    if not clone_dir:
        return {
            **state,
            'verification_results': {
                'ran_any': False,
                'note': 'No clone directory available for verification.',
                'results': [],
                'overall_pass': None,
            },
        }
    try:
        results = run_best_effort_verification(clone_dir, metadata)
        return {
            **state,
            'verification_results': results,
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"Verification complete. Overall pass: {results.get('overall_pass', 'N/A')}")
            ],
        }
    except Exception as e:
        return {
            **state,
            'verification_results': {
                'ran_any': False,
                'note': f'Verification failed: {e}',
                'results': [],
                'overall_pass': None,
            },
            'error': f'Verification failed: {e}',
        }


def prepare_pr_node(state: WorkflowState) -> WorkflowState:
    """Prepare PR title and body from implementation results."""
    proposal = state.get('feature_proposal', {})
    plan = state.get('implementation_plan', '')
    changed_files = state.get('changed_files', [])
    diff_summary = state.get('diff_summary', '')
    verification = state.get('verification_results', {})
    issue_url = state.get('issue_url', '')
    issue_number = state.get('issue_number')

    pr_title = f"feat: {proposal.get('title', 'Feature implementation')}"

    body_lines = [
        '## Summary',
        '',
        proposal.get('body', ''),
        '',
        '## Changes',
        '',
    ]
    for f in changed_files:
        body_lines.append(f'- `{f}`')
    body_lines.append('')

    body_lines.append('## Verification')
    if verification.get('ran_any'):
        body_lines.append(f"Ran {len(verification.get('results', []))} command(s).")
        body_lines.append(f"Overall pass: **{'Yes' if verification.get('overall_pass') else 'No'}**")
        for r in verification.get('results', []):
            body_lines.append(f"- `{r['command']}` — {r['status']} (exit {r['returncode']})")
    else:
        body_lines.append(verification.get('note', 'No verification run.'))
    body_lines.append('')

    body_lines.append('## Linked Issue')
    if issue_number and issue_url:
        body_lines.append(f'Closes #{issue_number}')
        body_lines.append(f'{issue_url}')
    else:
        body_lines.append('No linked issue.')
    body_lines.append('')

    body_lines.append('---')
    body_lines.append('*This pull request was created by the Autonomous Code Writer workflow.*')
    body_lines.append('*The human user retains final authority over merge decisions.*')

    pr_body = '\n'.join(body_lines)

    return {
        **state,
        'pr_title': pr_title,
        'pr_body': pr_body,
        'messages': state.get('messages', []) + [
            HumanMessage(content='PR metadata prepared.')
        ],
    }


def implementation_summary_node(state: WorkflowState) -> WorkflowState:
    """Display the implementation summary and collect human decision for Gate 2."""
    proposal = state.get('feature_proposal', {})
    plan = state.get('implementation_plan', '')
    changed_files = state.get('changed_files', [])
    diff_summary = state.get('diff_summary', '')
    verification = state.get('verification_results', {})
    pr_title = state.get('pr_title', '')

    print('\n' + '=' * 60)
    print('IMPLEMENTATION SUMMARY — HUMAN REVIEW REQUIRED')
    print('=' * 60)
    print(f"Proposed PR title: {pr_title}")
    print('')
    print('Implementation plan:')
    print(plan[:500] + ('...' if len(plan) > 500 else ''))
    print('')
    print('Changed files:')
    for f in changed_files:
        print(f'  - {f}')
    print('')
    print('Verification:')
    if verification.get('ran_any'):
        print(f"  Ran {len(verification.get('results', []))} command(s)")
        print(f"  Overall pass: {verification.get('overall_pass')}")
        for r in verification.get('results', []):
            print(f"    - {r['command']} -> {r['status']} (exit {r['returncode']})")
    else:
        print(f"  {verification.get('note', 'No verification run')}")
    print('')
    print('Diff summary (first 800 chars):')
    print(diff_summary[:800] + ('...' if len(diff_summary) > 800 else ''))
    print('')
    no_changes = not changed_files
    if no_changes:
        print('Status: blocked — no code changes were produced, so PR creation is disabled.')
        print('')
    print('-' * 60)
    print('Type your decision:')
    print('  approve  — push branch and create pull request')
    print('  reject   — stop the workflow')
    print('  revise <feedback> — update implementation before PR')
    print('=' * 60 + '\n')

    try:
        user_input = input('Your decision: ').strip()
    except EOFError:
        user_input = ''

    decision = 'pending'
    feedback = ''
    if user_input.lower() == 'approve':
        if no_changes:
            decision = 'revise'
            feedback = 'PR creation is blocked because the previous attempt produced no code changes. Apply a concrete diff before approval.'
        else:
            decision = 'approved'
    elif user_input.lower() == 'reject':
        decision = 'rejected'
    elif user_input.lower().startswith('revise'):
        decision = 'revise'
        feedback = user_input[6:].strip()
    else:
        decision = 'revise'
        feedback = user_input

    return {
        **state,
        'human_decision_gate2': {
            'decision': decision,
            'feedback': feedback,
            'timestamp': None,
        },
        'messages': state.get('messages', []) + [
            HumanMessage(content=f'Human gate 2 decision: {decision}')
        ],
    }


def route_gate2_decision(state: WorkflowState) -> str:
    """Route based on Gate 2 human decision."""
    decision = state.get('human_decision_gate2', {}).get('decision', 'pending')
    if decision == 'approved':
        return 'approve'
    elif decision == 'rejected':
        return 'reject'
    else:
        return 'revise'


def create_pr_node(state: WorkflowState) -> WorkflowState:
    """Push the branch and create a GitHub pull request."""
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    clone_dir = state.get('repo_clone_dir')
    branch_name = state.get('branch_name', '')
    pr_title = state.get('pr_title', '')
    pr_body = state.get('pr_body', '')
    changed_files = state.get('changed_files', [])
    repo_url = state.get('repo_url', '')

    if not clone_dir or not branch_name:
        return {
            **state,
            'error': 'Missing clone directory or branch name for PR creation.',
        }

    if not pr_title.strip():
        return {
            **state,
            'error': 'Missing PR title; PR metadata was not prepared correctly.',
        }

    if not changed_files:
        return {
            **state,
            'error': 'Refusing to create a pull request because no file changes were applied on the branch.',
        }

    try:
        # Push branch to remote
        push_branch(clone_dir, branch_name, repo_url, GITHUB_TOKEN)

        # Determine base branch
        repo = Repo(clone_dir)
        try:
            base_branch = repo.git.symbolic_ref('refs/remotes/origin/HEAD').replace('refs/remotes/origin/', '')
        except Exception:
            base_branch = 'main'

        # Create pull request
        pr_result = create_pull_request(owner, repo_name, pr_title, pr_body, branch_name, base_branch)

        return {
            **state,
            'pr_url': pr_result['pr_url'],
            'pr_number': pr_result['pr_number'],
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"Created PR #{pr_result['pr_number']}: {pr_result['pr_url']}")
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Failed to create pull request: {e}',
        }


def reject_pr_node(state: WorkflowState) -> WorkflowState:
    """Handle rejection at Gate 2 — stop the workflow gracefully."""
    print('\nWorkflow stopped: implementation was rejected at Gate 2.')
    return {
        **state,
        'messages': state.get('messages', []) + [
            HumanMessage(content='Workflow stopped: Gate 2 rejected.')
        ],
    }



In [ ]:
import subprocess
import shlex
from typing import Dict, Any, List

def run_shell_command(clone_dir: str, command: str, timeout: int = 120) -> Dict[str, Any]:
    result = {
        'command': command,
        'stdout': '',
        'stderr': '',
        'returncode': None,
        'status': 'unknown',
        'duration_ms': 0,
    }
    try:
        start = __import__('time').time()
        proc = subprocess.run(
            shlex.split(command),
            cwd=clone_dir,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        result['duration_ms'] = int((__import__('time').time() - start) * 1000)
        result['stdout'] = proc.stdout
        result['stderr'] = proc.stderr
        result['returncode'] = proc.returncode
        result['status'] = 'passed' if proc.returncode == 0 else 'failed'
    except subprocess.TimeoutExpired:
        result['status'] = 'timeout'
        result['stderr'] = f'Command timed out after {timeout}s'
    except FileNotFoundError:
        result['status'] = 'not_found'
        result['stderr'] = 'Command executable not found in environment'
    except Exception as e:
        result['status'] = 'error'
        result['stderr'] = str(e)
    return result


def run_best_effort_verification(clone_dir: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    test_commands = metadata.get('test_commands', [])
    lint_commands = metadata.get('lint_commands', [])
    commands = []
    if test_commands:
        commands.append(test_commands[0])
    if lint_commands:
        commands.append(lint_commands[0])
    if not commands:
        return {
            'ran_any': False,
            'note': 'No test or lint commands detected for this repository.',
            'results': [],
            'overall_pass': None,
        }

    pre_install = _pre_install_dependencies(clone_dir, metadata)

    results = []
    for cmd in commands:
        res = run_shell_command(clone_dir, cmd)
        max_out = 2000
        if len(res['stdout']) > max_out:
            res['stdout'] = res['stdout'][:max_out] + '\n... (truncated)\n'
        if len(res['stderr']) > max_out:
            res['stderr'] = res['stderr'][:max_out] + '\n... (truncated)\n'
        results.append(res)
    overall_pass = all(r['status'] == 'passed' for r in results)
    note = f'Ran {len(results)} verification command(s).'
    if pre_install:
        note += f' {pre_install}'
    return {
        'ran_any': True,
        'note': note,
        'results': results,
        'overall_pass': overall_pass,
    }


def _pre_install_dependencies(clone_dir: str, metadata: dict) -> str:
    package_managers = metadata.get('package_managers', [])
    if 'npm/yarn/pnpm' in package_managers or 'npm' in package_managers:
        if not os.path.exists(os.path.join(clone_dir, 'node_modules')):
            install_res = run_shell_command(clone_dir, 'npm install')
            if install_res['status'] == 'passed':
                return 'Ran npm install before verification.'
            else:
                return 'npm install failed; verification may fail due to missing dependencies.'
    if 'poetry' in package_managers:
        install_res = run_shell_command(clone_dir, 'poetry install')
        if install_res['status'] == 'passed':
            return 'Ran poetry install before verification.'
        else:
            return 'poetry install failed; verification may fail due to missing dependencies.'
    if 'pipenv' in package_managers:
        install_res = run_shell_command(clone_dir, 'pipenv install')
        if install_res['status'] == 'passed':
            return 'Ran pipenv install before verification.'
        else:
            return 'pipenv install failed; verification may fail due to missing dependencies.'
    if 'pip' in package_managers or 'setuptools' in package_managers or 'modern-python' in package_managers:
        req_path = os.path.join(clone_dir, 'requirements.txt')
        if os.path.exists(req_path):
            install_res = run_shell_command(clone_dir, 'pip install -r requirements.txt')
            if install_res['status'] == 'passed':
                return 'Ran pip install -r requirements.txt before verification.'
            else:
                return 'pip install failed; verification may fail due to missing dependencies.'
    return ''


In [ ]:
import json
from langchain_core.messages import SystemMessage, HumanMessage

PR_REVIEW_AGENT_PROMPT = """
You are a PR Review Agent. Your job is to review a pull request diff and produce a structured, honest assessment.

Review priorities (in order):
1. Correctness — Are there logic errors, off-by-one bugs, or incorrect assumptions?
2. Regressions — Could this change break existing functionality?
3. Security risks — Are there injection risks, unsafe defaults, or credential exposure?
4. Missing tests — Should new tests have been added or existing tests updated?
5. Maintainability — Is the code readable, consistent with project style, and appropriately documented?

Output format (strict JSON):
{
  "findings": [
    {
      "severity": "blocking|warning|note",
      "category": "correctness|regression|security|testing|maintainability",
      "description": "Clear description of the issue."
    }
  ],
  "residual_risks": "Any risks that remain even if the code looks correct.",
  "verification_notes": "Notes on whether tests or lint were present and what they covered.",
  "recommendation": "approve_with_minor_changes|approve|needs_revision|reject"
}

Guidelines:
- If no blocking issues are found, say so explicitly in findings (e.g., one finding with severity 'note' stating 'No blocking issues found').
- Be concise but specific. Cite file names or line patterns when possible.
- Do not invent issues that are not supported by the diff.
- The recommendation is advisory only; the human retains final merge authority.
"""

def build_pr_review_prompt(repo_owner: str, repo_name: str, pr_title: str, pr_body: str, diff_summary: str, changed_files: list, verification_results: dict) -> str:
    lines = [
        f'Repository: {repo_owner}/{repo_name}',
        f'PR title: {pr_title}',
        '',
        'PR body:',
        pr_body[:800],
        '',
        'Changed files:',
    ]
    for f in changed_files:
        lines.append(f'  - {f}')
    lines.append('')
    lines.append('Diff summary:')
    lines.append(diff_summary[:3000])
    lines.append('')
    ver = verification_results or {}
    if ver.get('ran_any'):
        lines.append('Verification results:')
        lines.append(f"  Overall pass: {ver.get('overall_pass')}")
        for r in ver.get('results', []):
            lines.append(f"  - {r['command']} -> {r['status']} (exit {r['returncode']})")
    else:
        lines.append(f"Verification: {ver.get('note', 'No verification run')}")
    return '\n'.join(lines)


def run_pr_review_agent(repo_owner: str, repo_name: str, pr_title: str, pr_body: str, diff_summary: str, changed_files: list, verification_results: dict) -> dict:
    prompt = build_pr_review_prompt(repo_owner, repo_name, pr_title, pr_body, diff_summary, changed_files, verification_results)
    messages = [
        SystemMessage(content=PR_REVIEW_AGENT_PROMPT),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    content = str(response.content)

    cleaned = content.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned[7:]
    if cleaned.startswith('```'):
        cleaned = cleaned[3:]
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        review = json.loads(cleaned)
    except json.JSONDecodeError:
        review = {
            'findings': [
                {
                    'severity': 'note',
                    'category': 'maintainability',
                    'description': 'No blocking issues found (review output could not be parsed).',
                }
            ],
            'residual_risks': 'Unable to assess residual risks due to parsing failure.',
            'verification_notes': 'Unable to assess verification coverage due to parsing failure.',
            'recommendation': 'approve',
        }
    return review


def format_review_for_github(review: dict) -> str:
    lines = [
        '## PR Review (Autonomous Code Writer)',
        '',
        'This review was generated automatically by the PR Review Agent.',
        'The human user retains final authority over merge decisions.',
        '',
        '---',
        '',
        '### Findings',
    ]
    findings = review.get('findings', [])
    if not findings:
        lines.append('- No findings reported.')
    for f in findings:
        severity = f.get('severity', 'note')
        category = f.get('category', 'general')
        description = f.get('description', '')
        lines.append(f"- **{severity.upper()}** ({category}): {description}")
    lines.append('')
    lines.append('### Residual Risks')
    lines.append(review.get('residual_risks', 'None noted.'))
    lines.append('')
    lines.append('### Verification Notes')
    lines.append(review.get('verification_notes', 'No verification notes.'))
    lines.append('')
    lines.append('### Recommendation')
    lines.append(f"**{review.get('recommendation', 'review').upper()}**")
    lines.append('')
    lines.append('---')
    lines.append('*This comment was posted by the Autonomous Code Writer workflow.*')
    return '\n'.join(lines)


In [ ]:
from github import GithubException

def post_pr_review_comment(owner: str, repo_name: str, pr_number: int, body: str) -> dict:
    """Post a review comment on a GitHub pull request. Returns the comment URL."""
    repo = github_client.get_repo(f'{owner}/{repo_name}')
    pr = repo.get_pull(pr_number)
    issue = pr.as_issue()
    comment = issue.create_comment(body)
    return {
        'comment_url': comment.html_url,
        'comment_id': comment.id,
    }


def retrieve_pr_diff(owner: str, repo_name: str, pr_number: int) -> str:
    """Retrieve the diff of a pull request from GitHub."""
    repo = github_client.get_repo(f'{owner}/{repo_name}')
    pr = repo.get_pull(pr_number)
    try:
        diff_text = pr.get_diff()
        return diff_text
    except Exception:
        return ''


In [ ]:
def pr_review_node(state: WorkflowState) -> WorkflowState:
    """Run the PR Review Agent to inspect the diff and produce a structured review."""
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    pr_title = state.get('pr_title', '')
    pr_body = state.get('pr_body', '')
    diff_summary = state.get('diff_summary', '')
    changed_files = state.get('changed_files', [])
    verification = state.get('verification_results', {})

    if not diff_summary and not changed_files:
        return {
            **state,
            'review_summary': 'No diff or changed files available for review.',
            'messages': state.get('messages', []) + [
                HumanMessage(content='PR Review Agent: no diff available.')
            ],
        }

    try:
        review = run_pr_review_agent(owner, repo_name, pr_title, pr_body, diff_summary, changed_files, verification)
        review_text = format_review_for_github(review)
        return {
            **state,
            'review_summary': review_text,
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"PR Review Agent produced {len(review.get('findings', []))} finding(s). Recommendation: {review.get('recommendation', 'review')}")
            ],
        }
    except Exception as e:
        return {
            **state,
            'review_summary': f'PR review failed: {e}',
            'messages': state.get('messages', []) + [
                HumanMessage(content=f'PR Review Agent failed: {e}')
            ],
        }


def post_review_comment_node(state: WorkflowState) -> WorkflowState:
    """Post the review summary as a GitHub PR comment."""
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    pr_number = state.get('pr_number')
    review_summary = state.get('review_summary', '')

    if not pr_number or not review_summary:
        return {
            **state,
            'review_comment_url': '',
            'messages': state.get('messages', []) + [
                HumanMessage(content='No PR number or review summary available; skipping comment post.')
            ],
        }

    try:
        result = post_pr_review_comment(owner, repo_name, pr_number, review_summary)
        return {
            **state,
            'review_comment_url': result['comment_url'],
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"Posted review comment: {result['comment_url']}")
            ],
        }
    except Exception as e:
        return {
            **state,
            'review_comment_url': '',
            'messages': state.get('messages', []) + [
                HumanMessage(content=f'Failed to post review comment: {e}')
            ],
        }


def final_output_node(state: WorkflowState) -> WorkflowState:
    """Produce the final workflow output with all collected artifacts."""
    print('\n' + '=' * 60)
    print('WORKFLOW COMPLETE')
    print('=' * 60)
    print(f"Repository: {state.get('repo_owner')}/{state.get('repo_name')}")
    print(f"Issue: {state.get('issue_url', 'N/A')}")
    print(f"Branch: {state.get('branch_name', 'N/A')}")
    print(f"Pull Request: {state.get('pr_url', 'N/A')}")
    print(f"PR Review Comment: {state.get('review_comment_url', 'N/A')}")
    if state.get('error'):
        print(f"Error: {state.get('error')}")
    print('\nPR Review Summary:')
    review = state.get('review_summary', 'No review generated.')
    print(review[:1000] + ('...' if len(review) > 1000 else ''))
    print('\nVerification Results:')
    ver = state.get('verification_results', {})
    if ver.get('ran_any'):
        print(f"  Overall pass: {ver.get('overall_pass')}")
        for r in ver.get('results', []):
            print(f"  - {r['command']} -> {r['status']}")
    else:
        print(f"  {ver.get('note', 'No verification available')}")
    print('=' * 60 + '\n')
    return {
        **state,
        'messages': state.get('messages', []) + [
            HumanMessage(content='Workflow completed successfully.')
        ],
    }


In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command


def build_workflow_graph() -> StateGraph:
    builder = StateGraph(WorkflowState)

    builder.add_node('parse_request', parse_request_node)
    builder.add_node('clone_and_inspect', clone_and_inspect_node)
    builder.add_node('analyze_repository', analyze_repository_node)
    builder.add_node('generate_feature_proposal', generate_feature_proposal_node)
    builder.add_node('human_review_gate1', human_review_gate1_node)
    builder.add_node('revise_feature_proposal', revise_feature_proposal_node)
    builder.add_node('create_issue', create_issue_node)
    builder.add_node('create_branch', create_branch_node)
    builder.add_node('implement', implementation_node)
    builder.add_node('verify', verification_node)
    builder.add_node('prepare_pr', prepare_pr_node)
    builder.add_node('implementation_summary', implementation_summary_node)
    builder.add_node('create_pr', create_pr_node)
    builder.add_node('reject_pr', reject_pr_node)
    builder.add_node('pr_review', pr_review_node)
    builder.add_node('post_review_comment', post_review_comment_node)
    builder.add_node('final_output', final_output_node)

    builder.set_entry_point('parse_request')

    builder.add_edge('parse_request', 'clone_and_inspect')
    builder.add_edge('clone_and_inspect', 'analyze_repository')
    builder.add_edge('analyze_repository', 'generate_feature_proposal')
    builder.add_edge('generate_feature_proposal', 'human_review_gate1')

    builder.add_conditional_edges(
        'human_review_gate1',
        route_gate1_decision,
        {
            'approve': 'create_issue',
            'reject': END,
            'revise': 'revise_feature_proposal',
        },
    )
    builder.add_edge('revise_feature_proposal', 'human_review_gate1')

    builder.add_edge('create_issue', 'create_branch')
    builder.add_edge('create_branch', 'implement')
    builder.add_edge('implement', 'verify')
    builder.add_edge('verify', 'prepare_pr')
    builder.add_edge('prepare_pr', 'implementation_summary')

    builder.add_conditional_edges(
        'implementation_summary',
        route_gate2_decision,
        {
            'approve': 'create_pr',
            'reject': 'reject_pr',
            'revise': 'implement',
        },
    )
    builder.add_edge('reject_pr', END)

    builder.add_edge('create_pr', 'pr_review')
    builder.add_edge('pr_review', 'post_review_comment')
    builder.add_edge('post_review_comment', 'final_output')
    builder.add_edge('final_output', END)

    return builder


In [ ]:
from IPython.display import Image, display

builder = build_workflow_graph()
graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
def execute_workflow(user_request: str) -> WorkflowState:
    """Run the autonomous code writer workflow for a single user request.

    The function initializes the LangGraph workflow with a memory checkpointer,
    starts execution, detects interruptions at human-in-the-loop gates,
    collects human feedback, resumes execution, and returns the final state.
    """
    builder = build_workflow_graph()
    checkpointer = MemorySaver()
    graph = builder.compile(checkpointer=checkpointer)

    thread_id = f'acw_{__import__("time").time():.0f}'
    config = {'configurable': {'thread_id': thread_id}}

    initial_state: WorkflowState = {
        'user_request': user_request,
        'messages': [],
    }

    current_state = initial_state
    iteration = 0
    max_iterations = 20

    while iteration < max_iterations:
        iteration += 1
        try:
            result = graph.invoke(current_state, config=config)
            current_state = result
            break
        except Exception as exc:
            exc_str = str(exc)
            if 'interrupt' in exc_str.lower() or 'human' in exc_str.lower() or 'input' in exc_str.lower():
                print(f'\n[Workflow paused — waiting for human input]')
                print(f'Error detail: {exc_str}')
                user_input = input('Provide input to resume (or press Enter to stop): ').strip()
                if not user_input:
                    print('Workflow stopped by empty input.')
                    break
                current_state = Command(resume=user_input)
                continue
            print(f'\n[Workflow error — stopping]')
            print(f'Error: {exc_str}')
            current_state = {
                **(current_state if isinstance(current_state, dict) else {}),
                'error': exc_str,
            }
            break

    if iteration >= max_iterations:
        print('Warning: workflow reached max iterations without completing.')

    _display_state_progress(current_state)
    return current_state

def _display_state_progress(state: WorkflowState) -> None:
    """Print a concise summary of the current workflow state progress."""
    print("\n" + "-" * 50)
    print("Workflow State Progress")
    print("-" * 50)
    print(f"Repository: {state.get('repo_owner')}/{state.get('repo_name')}")
    print(f"Summary: {'Present' if state.get('repo_summary') else 'Missing'}")
    print(f"Proposal: {state.get('feature_proposal', {}).get('title', 'N/A')}")
    print(f"Gate 1: {state.get('human_decision_gate1', {}).get('decision', 'N/A')}")
    print(f"Issue: {state.get('issue_url', 'N/A')}")
    print(f"Branch: {state.get('branch_name', 'N/A')}")
    print(f"Changed files: {len(state.get('changed_files', []))}")
    print(f"Verification: {state.get('verification_results', {}).get('overall_pass', 'N/A')}")
    print(f"Gate 2: {state.get('human_decision_gate2', {}).get('decision', 'N/A')}")
    print(f"PR: {state.get('pr_url', 'N/A')}")
    print(f"Review comment: {state.get('review_comment_url', 'N/A')}")
    print(f"Error: {state.get('error', 'N/A')}")
    print("-" * 50)


---

## Test case overview

| # | Scenario | Expected human responses |
|---|----------|--------------------------|
| 1 | Happy path — approve feature, approve PR | `approve` at Gate 1, `approve` at Gate 2 |
| 2 | Feature proposal revision before issue creation | `revise Make it a documentation improvement` at Gate 1, then `approve`, then `approve` at Gate 2 |
| 3 | Implementation revision before PR creation | `approve` at Gate 1, `revise Add more tests` at Gate 2, then `approve` |
| 4 | Documentation / developer-experience improvement | `approve` at Gate 1, `approve` at Gate 2 |
| 5 | Maintainability / validation improvement | `approve` at Gate 1, `approve` at Gate 2 |

Each test case creates a separate GitHub issue and branch. When credentials are configured, the workflow produces real issues, branches, pull requests, and review comments.

In [ ]:
DEMO_REPO_URL = 'https://github.com/BozhidarN7/ds-algo-visualizer'

print(f'Demo repository configured: {DEMO_REPO_URL}')
print('All test cases below use this repository. Replace the URL with your own repo if you want.')

## Test Case 1: Happy path — approve feature and approve PR

**Request:** Ask the system to analyze the repository and suggest a useful improvement.

**Expected human responses:**
- Gate 1: `approve`
- Gate 2: `approve`

**What it demonstrates:** The full happy path from repository analysis through issue creation, implementation, verification, PR creation, and automated review.

In [ ]:
result_1 = execute_workflow(
    f'Analyze this repository and suggest a useful improvement: {DEMO_REPO_URL}'
)

print('\n=== Test Case 1 Result ===')
print(f"Issue: {result_1.get('issue_url', 'N/A')}")
print(f"Branch: {result_1.get('branch_name', 'N/A')}")
print(f"PR: {result_1.get('pr_url', 'N/A')}")
print(f"Review comment: {result_1.get('review_comment_url', 'N/A')}")

## Test Case 2: Revise the feature proposal before issue creation

**Request:** Ask the system to analyze the repository and suggest a useful improvement, then revise the initial proposal.

**Expected human responses:**
- Gate 1: `revise Make it a documentation-focused improvement with setup instructions`
- Gate 1 (after revision): `approve`
- Gate 2: `approve`

**What it demonstrates:** The Human-in-the-Loop revision loop at Gate 1. The agent regenerates the feature proposal based on feedback before creating the GitHub issue.

In [ ]:
result_2 = execute_workflow(
    f'Analyze this repository and suggest a useful improvement: {DEMO_REPO_URL}'
)

print('\n=== Test Case 2 Result ===')
print(f"Issue: {result_2.get('issue_url', 'N/A')}")
print(f"Branch: {result_2.get('branch_name', 'N/A')}")
print(f"PR: {result_2.get('pr_url', 'N/A')}")
print(f"Review comment: {result_2.get('review_comment_url', 'N/A')}")

## Test Case 3: Revise the implementation before PR creation

**Request:** Ask the system to add a small developer-experience improvement.

**Expected human responses:**
- Gate 1: `approve`
- Gate 2: `revise Add unit tests for the new functionality`
- Gate 2 (after revision): `approve`

**What it demonstrates:** The Human-in-the-Loop revision loop at Gate 2. The agent re-implements the change based on feedback before pushing the branch and creating the pull request.

In [ ]:
result_3 = execute_workflow(
    f'Add a small developer-experience improvement to this repository: {DEMO_REPO_URL}'
)

print('\n=== Test Case 3 Result ===')
print(f"Issue: {result_3.get('issue_url', 'N/A')}")
print(f"Branch: {result_3.get('branch_name', 'N/A')}")
print(f"PR: {result_3.get('pr_url', 'N/A')}")
print(f"Review comment: {result_3.get('review_comment_url', 'N/A')}")

## Test Case 4: Documentation or developer-experience improvement

**Request:** Ask the system to improve project documentation or contributor onboarding.

**Expected human responses:**
- Gate 1: `approve`
- Gate 2: `approve`

**What it demonstrates:** A documentation-focused feature path. The agent proposes a safe, low-risk change such as improving README, adding a CONTRIBUTING guide, or creating setup instructions.

In [ ]:
result_4 = execute_workflow(
    f'Improve the documentation or contributor onboarding for this repository: {DEMO_REPO_URL}'
)

print('\n=== Test Case 4 Result ===')
print(f"Issue: {result_4.get('issue_url', 'N/A')}")
print(f"Branch: {result_4.get('branch_name', 'N/A')}")
print(f"PR: {result_4.get('pr_url', 'N/A')}")
print(f"Review comment: {result_4.get('review_comment_url', 'N/A')}")

## Test Case 5: Maintainability or validation improvement

**Request:** Ask the system to add tests, linting configuration, or other maintainability improvements.

**Expected human responses:**
- Gate 1: `approve`
- Gate 2: `approve`

**What it demonstrates:** A maintainability-focused feature path. The agent proposes adding missing tests, a basic test harness, or lint configuration to improve long-term project health.

In [ ]:
result_5 = execute_workflow(
    f'Add tests or improve maintainability for this repository: {DEMO_REPO_URL}'
)

print('\n=== Test Case 5 Result ===')
print(f"Issue: {result_5.get('issue_url', 'N/A')}")
print(f"Branch: {result_5.get('branch_name', 'N/A')}")
print(f"PR: {result_5.get('pr_url', 'N/A')}")
print(f"Review comment: {result_5.get('review_comment_url', 'N/A')}")

---

## Instructions

This section documents how to use the system safely, what the prototype intentionally does not do, and how to prepare the final assignment submission.

## API Keys and Secrets

The notebook requires two secrets:

- `OPENAI_API_KEY` — for the LLM calls (Repository Analyst, Feature Strategist, Implementation Agent, PR Review Agent).
- `GITHUB_TOKEN` — for all GitHub operations (issue creation, branch push, pull request creation, review comment posting).

**How to provide them:**

1. **Google Colab (recommended for grading):** Open the left sidebar, go to **Secrets** (key icon), and add `OPENAI_API_KEY` and `GITHUB_TOKEN`. The notebook reads them automatically via `google.colab.userdata`.
2. **Local development:** Create a `.env` file in the same directory as the notebook with:

   ```
   OPENAI_API_KEY=sk-...
   GITHUB_TOKEN=ghp_...
   ```

   The notebook loads it via `python-dotenv` in the second code cell.
3. **Environment variables:** Export the variables in your shell before starting Jupyter.

**Security rules:**

- Never paste API keys directly into notebook cells.
- Never commit the `.env` file or the notebook with keys inside it.
- The notebook does not print secret values.
- The notebook validates that keys are present without exposing them in output.

## GitHub Token Permissions

The `GITHUB_TOKEN` must have permission to read and write contents, issues, and pull requests for the target repository.

**Fine-grained token (recommended):**

| Permission | Access |
|------------|--------|
| Contents   | Read and write |
| Issues     | Read and write |
| Pull requests | Read and write |
| Metadata   | Read-only |

**Classic token:** The `repo` scope must be sufficient for issue, branch, commit, PR, and comment operations.

The workflow targets repositories you own or where you have write access. It does not support public repositories where you lack write permissions.

## Safety Guarantees

The workflow is designed with the following hard safety rules:

- **Never pushes to the default branch.** All implementation work happens on a new branch named `feature/acw-{issue-number}-{slug}`.
- **Never merges pull requests.** The workflow creates a PR and posts a review comment, but the final merge decision is always left to the human user.
- **Never creates GitHub artifacts without approval.** Issues and branches are created only after explicit human approval at Gate 1. Pull requests are created only after explicit human approval at Gate 2.
- **Never modifies secrets or deployment files.** The Implementation Agent has a safety filter that blocks changes to `.git/`, `.env`, CI/CD configs, deployment manifests, and credential files.
- **Human retains final authority.** The system is an assistant, not an autonomous actor. Every write action requires human consent.

## Known Limitations

This prototype is intentionally constrained to remain safe and understandable for the assignment. The following are out of scope:

- **No web UI.** The interface is the Jupyter notebook itself.
- **No webhook handling.** The workflow does not react to GitHub events in real time.
- **No auto-merge.** Pull requests are never merged automatically.
- **No production-grade sandboxing.** The agent runs in the same Colab or local environment as the notebook. It does not use isolated containers or VMs.
- **No multi-repository orchestration.** Each workflow run targets a single repository.
- **No long-running background agents.** The workflow runs synchronously inside the notebook kernel.
- **No automatic deployment or release creation.** The workflow stops after PR creation and review.
- **No large-scale refactoring.** The Implementation Agent prefers small, focused changes.
- **No database migrations or authentication rewrites.** These are blocked by the safety filter.
- **No full inline code review.** The PR Review Agent posts a single structured comment, not line-by-line annotations.
- **Best-effort verification only.** The system detects likely test and lint commands from file patterns, but it cannot guarantee that the right commands are run or that all edge cases are covered.

## Google Colab Setup and Execution Guide

### Step 1: Open the notebook in Colab

Upload `autonomous-code-writer.ipynb` to Google Drive, then open it with **Open with > Google Colaboratory**. Alternatively, open it directly from GitHub if the file is in a public repository.

### Step 2: Configure secrets

In the left sidebar, click the **key icon** (Secrets), and add:

- `OPENAI_API_KEY` — your OpenAI API key
- `GITHUB_TOKEN` — your GitHub personal access token

Make sure the token has the permissions listed above.

### Step 3: Set the demo repository

Replace the `DEMO_REPO_URL` in the test cases section with a repository you own and have write access to. The default is `https://github.com/BozhidarN7/ds-algo-visualizer`;

### Step 4: Run the cells in order

Use **Runtime > Run all** or run cells sequentially with `Shift+Enter`. The notebook installs dependencies, loads secrets, initializes the model and GitHub client, defines all agents and tools, builds the workflow graph, and then runs the test cases.

### Step 5: Respond at human approval gates

When the workflow pauses at Gate 1 or Gate 2, type one of the supported responses in the input box that appears below the cell:

- `approve` — proceed
- `reject` — stop the workflow
- `revise <feedback>` — regenerate the proposal or re-implement with your feedback

### Step 6: Inspect the results

After the workflow completes, the final cell prints the issue URL, branch name, PR URL, and review comment URL. You can open these links in GitHub to inspect the real artifacts created by the agent.


## Final Notes

This project is a **balanced prototype** built for the *AI Agents and Workflows for Developers* course assignment. It demonstrates a realistic multi-agent GitHub automation workflow using LangGraph and LangChain, while remaining safe, understandable, and grader-friendly.

The system is not a production-grade autonomous coding platform. It is intentionally constrained to:

- Small-to-medium feature changes on owned repositories.
- Branch-based implementation with human approval gates.
- Best-effort verification and structured PR review.

Future extensions could add sandboxed execution, inline code review comments, webhook triggers, or a web dashboard, but those are out of scope for the assignment version.

**The human user always retains final authority over merge decisions.**